Data Acquisition & Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/balanced_log_dataset.csv")

def parse_event_sequence(x):
    if pd.isna(x):
        return []
    x = x.strip().strip("[]")
    return [e.strip() for e in x.split(",") if e.strip()]

df['Features'] = df['Features'].apply(parse_event_sequence)

df['Label'] = df['Label'].map({'Success': 0, 'Fail': 1})
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Latency_norm'] = scaler.fit_transform(df[['Latency']])


print(df['Features'].head())
print(df['Label'].value_counts())


0    [E5, E5, E5, E22, E11, E9, E11, E9, E11, E9, E...
1    [E22, E5, E5, E5, E26, E26, E11, E9, E11, E9, ...
2    [E5, E5, E22, E5, E11, E9, E11, E9, E11, E9, E...
3    [E5, E5, E22, E5, E11, E9, E11, E9, E26, E26, ...
4    [E5, E5, E22, E5, E9, E11, E9, E11, E9, E26, E...
Name: Features, dtype: object
Label
1    16838
0    16838
Name: count, dtype: int64


In [ ]:
all_events = set(e for seq in df['Features'] for e in seq)

event2idx = {event: idx + 1 for idx, event in enumerate(all_events)}
event2idx['PAD'] = 0

vocab_size = len(event2idx)


In [ ]:
MAX_LEN = 50

def encode_sequence(seq):
    seq = [event2idx[e] for e in seq]
    return seq[:MAX_LEN] + [0] * max(0, MAX_LEN - len(seq))

df['encoded'] = df['Features'].apply(encode_sequence)


Feature Engineering & Model Definition

In [ ]:
X = torch.tensor(df['encoded'].tolist(), dtype=torch.long)
latency = torch.tensor(df['Latency_norm'].values, dtype=torch.float)
y = torch.tensor(df['Label'].values, dtype=torch.long)

X_train, X_test, lat_train, lat_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X,
    latency,
    y,
    df.index,            # ⭐ THIS IS THE KEY
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_test_idx = idx_test   # save for export



In [ ]:
class LogDataset(Dataset):
    def __init__(self, X, latency, y):
        self.X = X
        self.latency = latency
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.latency[idx], self.y[idx]



In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        # +1 for latency
        self.fc = nn.Linear(embed_dim + 1, 2)

    def forward(self, x, latency):
        # Padding mask (IMPORTANT)
        mask = (x == 0)

        x = self.embedding(x)
        x = self.encoder(x, src_key_padding_mask=mask)
        x = x.mean(dim=1)

        # concatenate latency
        x = torch.cat([x, latency.unsqueeze(1)], dim=1)

        return self.fc(x)

